# P07 — Kekeruhan Muara Sungai Maro dan Perairan Pesisir Merauke 2013–2025 dari Landsat 8/9 dan Sentinel-2

| Tahap | Isi | Waktu |
|---|---|---|
| 0–2 | Instalasi, login Earth Engine, Google Drive, konfigurasi | 2 menit |
| 3 | Muara Sungai Maro (akumulasi aliran HydroSHEDS) dan daerah aliran sungai (HydroBASINS level 8) | 1 menit |
| 4 | Kekeruhan bulanan (algoritma Dogliotti et al., 2015) dari Landsat 8/9 (2013–2025) dan Sentinel-2 (2019–2025), **disimpan di Drive** | 15–25 menit |
| 5 | Curah hujan bulanan DAS Maro (CHIRPS) | 1 menit |
| 6 | Kesesuaian Landsat vs Sentinel-2, zona jarak dari muara, musim hujan vs kemarau, luas plume | 2 menit |
| 7 | Tren 2014–2025 (Mann-Kendall + Sen) per zona dan per piksel | 2 menit |
| 8 | Hubungan dengan curah hujan (korelasi berjeda, Random Forest) dan zonasi rezim kekeruhan (k-means) | 2 menit |
| 9 | Gambar dua bahasa, unduh `P07_hasil.zip` | 2 menit |

**Tanpa survei lapangan dan tanpa penilaian manual.** *Runtime → Jalankan semua*. Bila Colab terputus, jalankan lagi; bulan yang sudah tersimpan di Drive tidak diunduh ulang.

In [ ]:
!pip install -q earthengine-api scikit-learn pandas matplotlib scipy openpyxl

In [ ]:
import os
GEE_PROJECT = 'riset-psn-pps'
try:
    import ee
    try:
        ee.Initialize(project=GEE_PROJECT)
    except Exception:
        ee.Authenticate(force=True); ee.Initialize(project=GEE_PROJECT)
    print('Earth Engine siap:', ee.String('OK').getInfo())
except Exception as e:
    print('Earth Engine tidak tersedia:', str(e)[:120])
try:
    from google.colab import drive; drive.mount('/content/drive'); DRIVE = '/content/drive/MyDrive'
except Exception:
    DRIVE = '.'
BASE = os.path.join(DRIVE, 'P07_kekeruhan'); os.makedirs(BASE, exist_ok=True); print('Folder kerja:', BASE)

In [ ]:
import json, time, zipfile, math, glob
import numpy as np, pandas as pd
TOPIC = 'P07'; SEED = 42
BBOX = [140.15, -8.80, 140.75, -8.30]            # pesisir Kota Merauke, muara Maro, dan Laut Arafura di depannya (sama dengan P01)
CITY = (140.39, -8.49)                           # titik bantu pencarian muara (Kota Merauke)
RES, CRS = 100, 'EPSG:32754'
L_START, S_START, END = '2013-04-01', '2019-01-01', '2026-01-01'
WET, DRY = [12, 1, 2, 3, 4], [6, 7, 8, 9, 10]  # musim hujan dan kemarau
ZONES = [(0, 5), (5, 15), (15, 30)]              # zona jarak dari muara (km)
MIN_VALID = 0.5                                  # bulan dipakai untuk luas plume bila >= 50% piksel zona 0–30 km punya data
A_RED, C_RED, A_NIR, C_NIR = 228.1, 0.1641, 3078.9, 0.2112   # Dogliotti et al. (2015), FNU
CACHE = os.path.join(BASE, 'cache'); os.makedirs(CACHE, exist_ok=True)
OUT = os.path.join(BASE, 'P07_hasil'); DIR_TAB, DIR_FIG, DIR_DATA = [os.path.join(OUT, d) for d in ('03_tabel', '02_gambar', '07_dataset')]
for d_ in (DIR_TAB, DIR_FIG, DIR_DATA): os.makedirs(d_, exist_ok=True)
ZNAME = {z: f'{z[0]}–{z[1]} km' for z in ZONES}
COLS_EN = {'tahun': 'year', 'citra_landsat': 'landsat_images', 'citra_sentinel2': 'sentinel2_images', 'bulan_landsat': 'landsat_months', 'bulan_sentinel2': 'sentinel2_months',
           'zona': 'zone', 'musim': 'season', 'n_bulan': 'n_months', 'median_FNU': 'median_FNU', 'p25_FNU': 'p25_FNU', 'p75_FNU': 'p75_FNU', 'rasio_hujan_kemarau': 'wet_dry_ratio',
           'nilai_p': 'p_value', 'besaran': 'quantity', 'nilai': 'value', 'tau_Kendall': 'Kendall_tau', 'Sen_FNU_per_thn': 'Sen_FNU_per_yr', 'Sen_persen_per_thn': 'Sen_pct_per_yr',
           'n_tahun': 'n_years', 'jeda_bulan': 'lag_months', 'variabel': 'variable', 'rho_Spearman': 'Spearman_rho', 'n': 'n', 'fitur': 'feature', 'kepentingan': 'importance',
           'skema': 'scheme', 'R2': 'R2', 'RMSE': 'RMSE', 'k': 'k', 'silhouette': 'silhouette', 'rezim': 'regime', 'luas_km2': 'area_km2', 'jarak_median_km': 'median_distance_km',
           'median_tahunan_FNU': 'annual_median_FNU', 'rasio_hujan_kemarau_rezim': 'regime_wet_dry_ratio', 'piksel_persen': 'pixels_pct', 'bulan': 'month', 'hujan_mm': 'rain_mm'}
VAL_EN = {'Hujan': 'Wet', 'Kemarau': 'Dry', 'Semua': 'All', 'Tinggal-satu-tahun': 'Leave-one-year-out'}
def save_tab(df, name_id, name_en):
    d = df.copy()
    for c in d.columns:
        if not pd.api.types.is_numeric_dtype(d[c]): d[c] = d[c].map(lambda v: VAL_EN.get(v, v))
    df.to_csv(os.path.join(DIR_TAB, f'P07_tabel_{name_id}_ID.csv'), index=False)
    d.rename(columns=COLS_EN).to_csv(os.path.join(DIR_TAB, f'P07_table_{name_en}_EN.csv'), index=False)
def utm_to_ll(x, y, zone=54, south=True):
    a, f, k0 = 6378137.0, 1 / 298.257223563, 0.9996; e2 = f * (2 - f); ep2 = e2 / (1 - e2)
    x = np.asarray(x, float) - 500000.0; y = np.asarray(y, float) - (10000000.0 if south else 0)
    M = y / k0; mu = M / (a * (1 - e2 / 4 - 3 * e2 ** 2 / 64 - 5 * e2 ** 3 / 256)); e1 = (1 - np.sqrt(1 - e2)) / (1 + np.sqrt(1 - e2))
    p1 = mu + (3 * e1 / 2 - 27 * e1 ** 3 / 32) * np.sin(2 * mu) + (21 * e1 ** 2 / 16 - 55 * e1 ** 4 / 32) * np.sin(4 * mu) + (151 * e1 ** 3 / 96) * np.sin(6 * mu)
    N1 = a / np.sqrt(1 - e2 * np.sin(p1) ** 2); T1 = np.tan(p1) ** 2; C1 = ep2 * np.cos(p1) ** 2; R1 = a * (1 - e2) / (1 - e2 * np.sin(p1) ** 2) ** 1.5; D = x / (N1 * k0)
    lat = p1 - (N1 * np.tan(p1) / R1) * (D ** 2 / 2 - (5 + 3 * T1 + 10 * C1 - 4 * C1 ** 2 - 9 * ep2) * D ** 4 / 24 + (61 + 90 * T1 + 298 * C1 + 45 * T1 ** 2 - 252 * ep2 - 3 * C1 ** 2) * D ** 6 / 720)
    lon = (D - (1 + 2 * T1 + C1) * D ** 3 / 6 + (5 - 2 * C1 + 28 * T1 - 3 * C1 ** 2 + 8 * ep2 + 24 * T1 ** 2) * D ** 5 / 120) / np.cos(p1)
    return np.degrees(lon) + (zone - 1) * 6 - 180 + 3, np.degrees(lat)
def ll_to_utm(lon, lat, zone=54, south=True):
    a, f, k0 = 6378137.0, 1 / 298.257223563, 0.9996; e2 = f * (2 - f); ep2 = e2 / (1 - e2)
    p = np.radians(np.asarray(lat, float)); l = np.radians(np.asarray(lon, float) - ((zone - 1) * 6 - 180 + 3))
    N = a / np.sqrt(1 - e2 * np.sin(p) ** 2); T = np.tan(p) ** 2; C = ep2 * np.cos(p) ** 2; A = np.cos(p) * l
    M = a * ((1 - e2 / 4 - 3 * e2 ** 2 / 64 - 5 * e2 ** 3 / 256) * p - (3 * e2 / 8 + 3 * e2 ** 2 / 32 + 45 * e2 ** 3 / 1024) * np.sin(2 * p)
             + (15 * e2 ** 2 / 256 + 45 * e2 ** 3 / 1024) * np.sin(4 * p) - (35 * e2 ** 3 / 3072) * np.sin(6 * p))
    x = k0 * N * (A + (1 - T + C) * A ** 3 / 6 + (5 - 18 * T + T ** 2 + 72 * C - 58 * ep2) * A ** 5 / 120) + 500000.0
    y = k0 * (M + N * np.tan(p) * (A ** 2 / 2 + (5 - T + 9 * C + 4 * C ** 2) * A ** 4 / 24 + (61 - 58 * T + T ** 2 + 600 * C - 330 * ep2) * A ** 6 / 720))
    return x, y + (10000000.0 if south else 0)
np.random.seed(SEED); print('Folder hasil:', OUT)

## Tahap 3 — Muara Sungai Maro dan daerah aliran sungainya
* **Muara:** piksel dengan akumulasi aliran terbesar (HydroSHEDS 15 detik busur) dalam radius 15 km dari Kota Merauke.
* **DAS Maro:** sub-DAS HydroBASINS level 8 di muara (luas hulu terbesar), lalu semua sub-DAS di hulunya ditelusuri melalui atribut `NEXT_DOWN`.

In [ ]:
F_META = os.path.join(CACHE, 'meta.json')
if os.path.exists(F_META):
    META = json.load(open(F_META))
else:
    rect = ee.Geometry.Rectangle(BBOX)
    ring = rect.transform(CRS, 1).bounds(1, CRS).coordinates().getInfo()[0]
    xs, ys = [p[0] for p in ring], [p[1] for p in ring]
    X0, Y0 = math.floor(min(xs) / RES) * RES, math.ceil(max(ys) / RES) * RES
    W, H = int(math.ceil((max(xs) - X0) / RES)), int(math.ceil((Y0 - min(ys)) / RES))
    acc = ee.Image('WWF/HydroSHEDS/15ACC').select('b1')
    mx = acc.addBands(ee.Image.pixelLonLat()).reduceRegion(ee.Reducer.max(3), ee.Geometry.Point(CITY).buffer(15000), 450).getInfo()
    mouth = (mx['max1'], mx['max2']); print('Muara Maro (akumulasi maksimum):', mouth, '| sel hulu:', mx['max'])
    hb = ee.FeatureCollection('WWF/HydroSHEDS/v1/Basins/hybas_8').filterBounds(ee.Geometry.Rectangle([139.5, -9.2, 141.5, -6.5]))
    tab = hb.reduceColumns(ee.Reducer.toList(3), ['HYBAS_ID', 'NEXT_DOWN', 'UP_AREA']).get('list').getInfo()
    near = hb.filterBounds(ee.Geometry.Point(mouth).buffer(5000)).sort('UP_AREA', False).first().get('HYBAS_ID').getInfo()
    down = {int(a): int(b) for a, b, _ in tab}; ids = []
    for h in down:
        k, seen = h, 0
        while k and k != near and k in down and seen < 500: k = down[k]; seen += 1
        if k == near: ids.append(h)
    catch = hb.filter(ee.Filter.inList('HYBAS_ID', ids)).union(1).geometry()
    area = catch.area(100).divide(1e6).getInfo(); print(f'DAS Maro: {len(ids)} sub-DAS, {area:,.0f} km²')
    json.dump({'X0': X0, 'Y0': Y0, 'W': W, 'H': H, 'mouth': mouth, 'acc_cells': mx['max'], 'catch_ids': ids, 'catch_km2': area,
               'catch_geojson': catch.simplify(500).getInfo()}, open(F_META, 'w'))
    META = json.load(open(F_META))
X0, Y0, W, H = META['X0'], META['Y0'], META['W'], META['H']
GRID = {'dimensions': {'width': W, 'height': H}, 'affineTransform': {'scaleX': RES, 'shearX': 0, 'translateX': X0, 'shearY': 0, 'scaleY': -RES, 'translateY': Y0}, 'crsCode': CRS}
json.dump({'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'properties': {'nama': 'DAS Maro (HydroBASINS lv8)', 'luas_km2': META['catch_km2']}, 'geometry': META['catch_geojson']},
           {'type': 'Feature', 'properties': {'nama': 'muara Maro'}, 'geometry': {'type': 'Point', 'coordinates': META['mouth']}}]}, open(os.path.join(DIR_DATA, 'P07_DAS_dan_muara.geojson'), 'w'))
print(f'Grid {W} x {H} piksel ({RES} m); muara {META["mouth"]}; DAS {META["catch_km2"]:,.0f} km²')

## Tahap 4 — Kekeruhan bulanan
Untuk setiap citra: awan dan bayangan dibuang (Landsat QA_PIXEL; Sentinel-2 Cloud Score+ ≥ 0,65), hanya piksel air saat perekaman (MNDWI > 0 dan kemunculan air JRC GSW ≥ 50%), reflektan dikurangi SWIR2 sebagai koreksi sisa atmosfer/glint, lalu kekeruhan dihitung dengan **algoritma switching Dogliotti et al. (2015)**:
T = A·ρ / (1 − ρ/C); merah (A = 228,1; C = 0,1641) bila ρ_merah < 0,05, NIR (A = 3078,9; C = 0,2112) bila ρ_merah > 0,07, campuran linear di antaranya. Median bulanan disimpan per bulan di Drive.

In [ ]:
def turb(red, nir):
    tr = red.multiply(A_RED).divide(ee.Image(1).subtract(red.divide(C_RED)))
    tn = nir.multiply(A_NIR).divide(ee.Image(1).subtract(nir.divide(C_NIR)))
    w = red.subtract(0.05).divide(0.02).clamp(0, 1)
    return tr.multiply(ee.Image(1).subtract(w)).add(tn.multiply(w)).rename('T')
def lsat(i):
    qa = i.select('QA_PIXEL'); ok = qa.bitwiseAnd(0b11110).eq(0)
    sr = i.select(['SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']).multiply(0.0000275).add(-0.2)
    g, r, n, s1, s2 = [sr.select(b) for b in ['SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']]
    wat = g.subtract(s1).divide(g.add(s1)).gt(0)
    t = turb(r.subtract(s2).max(0), n.subtract(s2).max(0))
    return t.updateMask(ok.And(wat).And(t.gte(0)).And(t.lt(5000)))
def s2(i):
    ok = i.select('cs_cdf').gte(0.65)
    sr = i.select(['B3', 'B4', 'B8A', 'B11', 'B12']).divide(10000)
    g, r, n, s1, s2_ = [sr.select(b) for b in ['B3', 'B4', 'B8A', 'B11', 'B12']]
    wat = g.subtract(s1).divide(g.add(s1)).gt(0)
    t = turb(r.subtract(s2_).max(0), n.subtract(s2_).max(0))
    return t.updateMask(ok.And(wat).And(t.gte(0)).And(t.lt(5000)))
def month_img(sensor, y, m):
    a = ee.Date.fromYMD(y, m, 1); b = a.advance(1, 'month'); rect = ee.Geometry.Rectangle(BBOX)
    gsw = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').unmask(0).gte(50)
    if sensor == 'L':
        col = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')).filterBounds(rect).filterDate(a, b).map(lsat)
    else:
        col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(rect).filterDate(a, b).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 90))
               .linkCollection(ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED'), ['cs_cdf']).map(s2))
    img = ee.Image.cat([col.median().updateMask(gsw).multiply(10).min(32000).unmask(-1).toInt16().rename('T'), col.count().unmask(0).toInt16().rename('n')])
    return img, col.size()
def fetch(img):
    out = {}
    for b in img.bandNames().getInfo():
        out[b] = ee.data.computePixels({'expression': img.select(b), 'fileFormat': 'NUMPY_NDARRAY', 'grid': GRID})[b]
    return out
MONTHS = {'L': pd.period_range(L_START, END, freq='M')[:-1], 'S': pd.period_range(S_START, END, freq='M')[:-1]}
NIMG = []
for sensor, months in MONTHS.items():
    t0 = time.perf_counter(); new = 0
    for p in months:
        fn = os.path.join(CACHE, f'{sensor}_{p.year}_{p.month:02d}.npz')
        if os.path.exists(fn):
            NIMG.append({'sensor': sensor, 'tahun': p.year, 'bulan': p.month, 'n': int(np.load(fn)['size'])}); continue
        for attempt in range(4):
            try:
                img, size = month_img(sensor, p.year, p.month); size = size.getInfo()
                arr = fetch(img) if size > 0 else {'T': np.full((H, W), -1, np.int16), 'n': np.zeros((H, W), np.int16)}
                np.savez_compressed(fn, size=size, **arr); NIMG.append({'sensor': sensor, 'tahun': p.year, 'bulan': p.month, 'n': size}); new += 1; break
            except Exception as ex:
                print(f'  {sensor} {p}: percobaan {attempt + 1} gagal ({str(ex)[:80]})'); time.sleep(15)
    print(f'{"Landsat 8/9" if sensor == "L" else "Sentinel-2"}: {len(months)} bulan ({new} baru, {time.perf_counter() - t0:.0f} dtk)')
NIMG = pd.DataFrame(NIMG)
assert all(os.path.exists(os.path.join(CACHE, f'{s}_{p.year}_{p.month:02d}.npz')) for s, ms in MONTHS.items() for p in ms), 'Ada bulan yang gagal. Jalankan ulang sel ini.'

## Tahap 5 — Curah hujan bulanan DAS Maro (CHIRPS)

In [ ]:
F_RAIN = os.path.join(CACHE, 'hujan.csv')
if not os.path.exists(F_RAIN):
    catch = ee.Geometry(META['catch_geojson'])
    CH = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
    def mrain(y, m):
        c = CH.filter(ee.Filter.calendarRange(y, y, 'year')).filter(ee.Filter.calendarRange(m, m, 'month'))
        v = c.sum().reduceRegion(ee.Reducer.mean(), catch, 5000).get('precipitation')
        return ee.Algorithms.If(c.size().gt(0), v, None)          # bulan tanpa data CHIRPS -> kosong
    rows = []
    for y in range(int(L_START[:4]), int(END[:4])):
        lst = ee.List.sequence(1, 12).map(lambda m: mrain(y, m)).getInfo()
        rows += [{'tahun': y, 'bulan': m + 1, 'hujan_mm': v} for m, v in enumerate(lst)]
    pd.DataFrame(rows).to_csv(F_RAIN, index=False)
RAIN = pd.read_csv(F_RAIN); RAIN['p'] = pd.PeriodIndex.from_fields(year=RAIN.tahun, month=RAIN.bulan, freq='M')
print(RAIN.groupby('bulan').hujan_mm.mean().round(0).to_dict())

## Tahap 6 — Kesesuaian sensor, zona, musim, dan plume
* **Zona jarak** dari muara: 0–5, 5–15, 15–30 km (hanya piksel air).
* **Kesesuaian Landsat vs Sentinel-2:** pada bulan dan piksel yang sama-sama berdata (2019–2025).
* **Musim:** hujan (Des–Apr) vs kemarau (Jun–Okt), uji Mann-Whitney pada median bulanan zona.
* **Plume:** ambang Otsu pada log10 kekeruhan seluruh piksel-bulan Landsat; persentase piksel 0–30 km di atas ambang pada bulan dengan cakupan ≥ 50%.

In [ ]:
from scipy import stats, ndimage as ndi
def load(sensor):
    ps = MONTHS[sensor]; A = np.full((len(ps), H, W), np.nan, np.float32)
    for k, p in enumerate(ps):
        a = np.load(os.path.join(CACHE, f'{sensor}_{p.year}_{p.month:02d}.npz'))['T'].astype(np.float32); a[a < 0] = np.nan; A[k] = a / 10.0
    return ps, A
PL, TL = load('L'); PS, TS = load('S')
mx_, my_ = ll_to_utm(*META['mouth']); cxm, cym = (mx_ - X0) / RES - 0.5, (Y0 - my_) / RES - 0.5
yy, xx = np.mgrid[0:H, 0:W]; DIST = np.hypot(xx - cxm, yy - cym) * RES / 1000
VALIDF = np.isfinite(TL).mean(0); WATER = VALIDF >= 0.3
ZMASK = {z: WATER & (DIST >= z[0]) & (DIST < z[1]) for z in ZONES}; Z30 = WATER & (DIST < ZONES[-1][1])
print({ZNAME[z]: int(m.sum()) for z, m in ZMASK.items()}, 'piksel air per zona')
def zone_series(ps, A):
    rows = []
    for k, p in enumerate(ps):
        r = {'p': p, 'tahun': p.year, 'bulan': p.month}
        for z, m in ZMASK.items():
            v = A[k][m]; v = v[np.isfinite(v)]; r[ZNAME[z]] = np.median(v) if len(v) >= 0.2 * m.sum() and len(v) > 0 else np.nan
        rows.append(r)
    return pd.DataFrame(rows)
ZL, ZS = zone_series(PL, TL), zone_series(PS, TS)
pd.concat([ZL.assign(sensor='Landsat 8/9'), ZS.assign(sensor='Sentinel-2')]).drop(columns='p').round(3).to_csv(os.path.join(DIR_DATA, 'P07_deret_bulanan_zona.csv'), index=False)
# data per tahun
DY = []
for y in range(PL[0].year, PL[-1].year + 1):
    nl = NIMG[(NIMG.sensor == 'L') & (NIMG.tahun == y)]; ns = NIMG[(NIMG.sensor == 'S') & (NIMG.tahun == y)]
    DY.append({'tahun': y, 'citra_landsat': int(nl.n.sum()), 'bulan_landsat': int(ZL[ZL.tahun == y][ZNAME[ZONES[0]]].notna().sum()),
               'citra_sentinel2': int(ns.n.sum()), 'bulan_sentinel2': int(ZS[ZS.tahun == y][ZNAME[ZONES[0]]].notna().sum())})
DY = pd.DataFrame(DY); save_tab(DY, 'data_per_tahun', 'data_per_year'); print(DY.to_string(index=False))
# kesesuaian sensor (bulan sama, piksel sama)
iL = {p: k for k, p in enumerate(PL)}; xs_, ys_ = [], []
for k, p in enumerate(PS):
    if p in iL:
        a, b = TL[iL[p]][Z30], TS[k][Z30]; ok = np.isfinite(a) & np.isfinite(b) & (a > 0) & (b > 0); xs_.append(a[ok]); ys_.append(b[ok])
xs_, ys_ = np.concatenate(xs_), np.concatenate(ys_)
rho = stats.spearmanr(xs_, ys_)[0] if len(xs_) > 10 else np.nan
CMP = pd.DataFrame([{'besaran': 'n pasangan piksel-bulan', 'nilai': len(xs_)}, {'besaran': 'rho Spearman', 'nilai': rho},
                    {'besaran': 'median rasio S2/Landsat', 'nilai': float(np.median(ys_ / xs_)) if len(xs_) else np.nan},
                    {'besaran': 'RMSE log10', 'nilai': float(np.sqrt(np.mean((np.log10(ys_) - np.log10(xs_)) ** 2))) if len(xs_) else np.nan},
                    {'besaran': 'rho Spearman median zona 0–5 km', 'nilai': stats.spearmanr(*ZL.merge(ZS, on='p')[[f'{ZNAME[ZONES[0]]}_x', f'{ZNAME[ZONES[0]]}_y']].dropna().values.T)[0]}]).round(4)
save_tab(CMP, 'kesesuaian_sensor', 'sensor_agreement'); print(CMP.to_string(index=False))
# musim
SEA = []
for z in ZONES:
    s = ZL[ZNAME[z]]; w_, d_ = s[ZL.bulan.isin(WET)].dropna(), s[ZL.bulan.isin(DRY)].dropna()
    for nm, v in (('Hujan', w_), ('Kemarau', d_)):
        SEA.append({'zona': ZNAME[z], 'musim': nm, 'n_bulan': len(v), 'median_FNU': v.median(), 'p25_FNU': v.quantile(0.25), 'p75_FNU': v.quantile(0.75),
                    'rasio_hujan_kemarau': w_.median() / d_.median() if nm == 'Hujan' and len(d_) else np.nan,
                    'nilai_p': stats.mannwhitneyu(w_, d_).pvalue if nm == 'Hujan' and len(w_) > 2 and len(d_) > 2 else np.nan})
SEA = pd.DataFrame(SEA).round(4); save_tab(SEA, 'musim_per_zona', 'season_by_zone'); print(SEA.to_string(index=False))
# plume
def otsu(v):
    v = v[np.isfinite(v)]; h, e = np.histogram(v, 256); c = (e[:-1] + e[1:]) / 2; w = np.cumsum(h); m = np.cumsum(h * c); wt, mt = w[-1], m[-1]
    b = (mt * w - m * wt) ** 2 / np.maximum(w * (wt - w), 1e-9); return float(c[np.nanargmax(b)])
lv = np.log10(np.clip(TL[:, Z30], 0.1, None)); sub = lv[np.isfinite(lv)]; sub = np.random.default_rng(SEED).choice(sub, size=min(2_000_000, len(sub)), replace=False)
T_PLUME = 10 ** otsu(sub); PLUME = []
for k, p in enumerate(PL):
    v = TL[k][Z30]; ok = np.isfinite(v); cov = ok.mean()
    PLUME.append({'p': p, 'tahun': p.year, 'bulan': p.month, 'cakupan': cov, 'plume_persen': 100 * float((v[ok] > T_PLUME).mean()) if cov >= MIN_VALID else np.nan})
PLUME = pd.DataFrame(PLUME); print(f'Ambang plume (Otsu) = {T_PLUME:.1f} FNU; bulan dengan cakupan cukup: {PLUME.plume_persen.notna().sum()}')

## Tahap 7 — Tren 2014–2025
Median tahunan (tahun penuh 2014–2025, Landsat) → **uji Mann-Kendall** dan **kemiringan Sen**, per zona dan per piksel (piksel dengan ≥ 10 tahun berdata). Tren juga dihitung untuk persentase plume tahunan.

In [ ]:
def mk(y):
    y = np.asarray(y, float); t = np.arange(len(y)); ok = np.isfinite(y); y, t = y[ok], t[ok]; n = len(y)
    if n < 6: return np.nan, np.nan, np.nan
    s = sum(np.sign(y[j] - y[i]) for i in range(n - 1) for j in range(i + 1, n)); var = n * (n - 1) * (2 * n + 5) / 18
    z = (s - np.sign(s)) / np.sqrt(var) if s != 0 else 0.0; p = 2 * (1 - stats.norm.cdf(abs(z))); tau = s / (n * (n - 1) / 2)
    sl = np.median([(y[j] - y[i]) / (t[j] - t[i]) for i in range(n - 1) for j in range(i + 1, n)]); return tau, p, sl
YRS = list(range(2014, 2026)); TR = []
AN = ZL[ZL.tahun.isin(YRS)].groupby('tahun')[[ZNAME[z] for z in ZONES]].median()
for z in ZONES:
    tau, p, sl = mk(AN[ZNAME[z]].reindex(YRS).values); med = AN[ZNAME[z]].median()
    TR.append({'zona': ZNAME[z], 'n_tahun': int(AN[ZNAME[z]].notna().sum()), 'tau_Kendall': tau, 'nilai_p': p, 'Sen_FNU_per_thn': sl, 'Sen_persen_per_thn': 100 * sl / med if med else np.nan})
PA = PLUME[PLUME.tahun.isin(YRS)].groupby('tahun').plume_persen.median().reindex(YRS); tau, p, sl = mk(PA.values)
TR.append({'zona': 'Plume (% piksel 0–30 km)', 'n_tahun': int(PA.notna().sum()), 'tau_Kendall': tau, 'nilai_p': p, 'Sen_FNU_per_thn': sl, 'Sen_persen_per_thn': np.nan})
TREND = pd.DataFrame(TR).round(4); save_tab(TREND, 'tren_per_zona', 'trend_by_zone'); print(TREND.to_string(index=False))
# per piksel
kk = [k for k, p in enumerate(PL) if p.year in YRS]; ANP = np.full((len(YRS), H, W), np.nan, np.float32)
for i, y in enumerate(YRS):
    idx = [k for k in kk if PL[k].year == y]
    with np.errstate(all='ignore'): ANP[i] = np.nanmedian(TL[idx], axis=0)
ok = WATER & (np.isfinite(ANP).sum(0) >= 10); V = ANP[:, ok]; n = len(YRS); S_ = np.zeros(V.shape[1]); slopes = []
for i in range(n - 1):
    for j in range(i + 1, n):
        d = V[j] - V[i]; S_ += np.nan_to_num(np.sign(d)); slopes.append(d / (j - i))
nn = np.isfinite(V).sum(0); var = nn * (nn - 1) * (2 * nn + 5) / 18; Z = np.where(S_ != 0, (S_ - np.sign(S_)) / np.sqrt(var), 0); P_ = 2 * (1 - stats.norm.cdf(np.abs(Z)))
with np.errstate(all='ignore'): SEN = np.nanmedian(np.array(slopes), axis=0)
SENMAP = np.full((H, W), np.nan, np.float32); SENMAP[ok] = SEN; SIGMAP = np.zeros((H, W), bool); SIGMAP[ok] = P_ < 0.05
PIXT = pd.DataFrame([{'besaran': 'piksel dinilai', 'nilai': int(ok.sum())}, {'besaran': 'naik signifikan (%)', 'nilai': 100 * float(((SEN > 0) & (P_ < 0.05)).mean())},
                     {'besaran': 'turun signifikan (%)', 'nilai': 100 * float(((SEN < 0) & (P_ < 0.05)).mean())}, {'besaran': 'median Sen (FNU/thn)', 'nilai': float(np.nanmedian(SEN))}]).round(3)
save_tab(PIXT, 'tren_per_piksel', 'pixel_trend'); print(PIXT.to_string(index=False))

## Tahap 8 — Hubungan dengan curah hujan dan zonasi rezim kekeruhan
* **Korelasi Spearman** median kekeruhan zona 0–5 km dan persentase plume dengan hujan DAS bulan yang sama dan 1–2 bulan sebelumnya.
* **Random Forest** (kekeruhan zona 0–5 km, log10) dari hujan jeda 0–2 bulan dan siklus musim (sin/cos bulan); validasi **tinggal-satu-tahun**; kepentingan permutasi.
* **k-means** pada siklus musiman per piksel (median log10 kekeruhan per bulan kalender), k = 2–6 dipilih dengan silhouette.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, r2_score
DF = ZL[['p', 'tahun', 'bulan', ZNAME[ZONES[0]]]].merge(PLUME[['p', 'plume_persen']], on='p').merge(RAIN[['p', 'hujan_mm']], on='p', how='left').sort_values('p').reset_index(drop=True)
for l in (1, 2): DF[f'hujan_jeda{l}'] = DF.hujan_mm.shift(l)
DF['hujan_jeda0'] = DF.hujan_mm; DF['y'] = np.log10(DF[ZNAME[ZONES[0]]].clip(lower=0.1))
COR = []
for tgt, nm in ((ZNAME[ZONES[0]], f'Kekeruhan {ZNAME[ZONES[0]]}'), ('plume_persen', 'Plume (%)')):
    for l in (0, 1, 2):
        d = DF[[tgt, f'hujan_jeda{l}']].dropna()
        if len(d) > 10: r, p = stats.spearmanr(d[tgt], d[f'hujan_jeda{l}']); COR.append({'variabel': nm, 'jeda_bulan': l, 'n': len(d), 'rho_Spearman': r, 'nilai_p': p})
COR = pd.DataFrame(COR).round(4); save_tab(COR, 'korelasi_hujan', 'rain_correlation'); print(COR.to_string(index=False))
DF['sin_bulan'], DF['cos_bulan'] = np.sin(2 * np.pi * DF.bulan / 12), np.cos(2 * np.pi * DF.bulan / 12)
FX = ['hujan_jeda0', 'hujan_jeda1', 'hujan_jeda2', 'sin_bulan', 'cos_bulan']; M = DF.dropna(subset=FX + ['y']).reset_index(drop=True)
pred = np.full(len(M), np.nan)
for y in M.tahun.unique():
    tr, te = M.tahun != y, M.tahun == y
    pred[te.values] = RandomForestRegressor(300, min_samples_leaf=3, random_state=SEED).fit(M.loc[tr, FX], M.y[tr]).predict(M.loc[te, FX])
rf = RandomForestRegressor(300, min_samples_leaf=3, random_state=SEED).fit(M[FX], M.y); base = r2_score(M.y, pred); rng = np.random.default_rng(SEED); imp = []
for f in FX:
    dr = []
    for _ in range(20):
        Mp = M.copy(); Mp[f] = rng.permutation(Mp[f].values); pp = np.full(len(M), np.nan)
        for y in M.tahun.unique():
            tr, te = M.tahun != y, M.tahun == y
            pp[te.values] = RandomForestRegressor(100, min_samples_leaf=3, random_state=SEED).fit(M.loc[tr, FX], M.y[tr]).predict(Mp.loc[te, FX])
        dr.append(base - r2_score(M.y, pp))
        if len(dr) >= 5: break
    imp.append({'fitur': f, 'kepentingan': float(np.mean(dr))})
RFT = pd.DataFrame([{'skema': 'Tinggal-satu-tahun', 'n': len(M), 'R2': base, 'RMSE': float(np.sqrt(np.mean((M.y - pred) ** 2)))}]).round(4)
save_tab(RFT, 'model_hujan_RF', 'rain_RF_model'); IMPT = pd.DataFrame(imp).round(4); save_tab(IMPT, 'kepentingan_RF', 'RF_importance'); print(RFT.to_string(index=False)); print(IMPT.to_string(index=False))
# k-means rezim
CLIM = np.full((12, H, W), np.nan, np.float32)
for m in range(1, 13):
    idx = [k for k, p in enumerate(PL) if p.month == m]
    with np.errstate(all='ignore'): CLIM[m - 1] = np.nanmedian(np.log10(np.clip(TL[idx], 0.1, None)), axis=0)
okc = WATER & (np.isfinite(CLIM).sum(0) >= 9); Xc = CLIM[:, okc].T; cm = np.nanmean(Xc, 1, keepdims=True); Xc = np.where(np.isfinite(Xc), Xc, cm)
smp = np.random.default_rng(SEED).choice(len(Xc), size=min(8000, len(Xc)), replace=False); SIL = []
for k in range(2, 7):
    km_ = KMeans(k, n_init=10, random_state=SEED).fit(Xc[smp]); SIL.append({'k': k, 'silhouette': silhouette_score(Xc[smp], km_.labels_)})
SIL = pd.DataFrame(SIL).round(4); KBEST = int(SIL.sort_values('silhouette', ascending=False).k.iloc[0]); save_tab(SIL, 'silhouette_kmeans', 'kmeans_silhouette')
km = KMeans(KBEST, n_init=10, random_state=SEED).fit(Xc); order = np.argsort(km.cluster_centers_.mean(1)); lab = np.argsort(order)[km.labels_] + 1
REG = np.zeros((H, W), np.uint8); REG[okc] = lab; RG = []
for r in range(1, KBEST + 1):
    m_ = REG == r; c = km.cluster_centers_[order[r - 1]]
    RG.append({'rezim': r, 'luas_km2': m_.sum() * RES * RES / 1e6, 'piksel_persen': 100 * m_.sum() / okc.sum(), 'jarak_median_km': float(np.median(DIST[m_])),
               'median_tahunan_FNU': float(10 ** np.median(c)), 'rasio_hujan_kemarau_rezim': float(10 ** (np.mean(c[[w - 1 for w in WET]]) - np.mean(c[[d - 1 for d in DRY]])))})
RG = pd.DataFrame(RG).round(3); save_tab(RG, 'rezim_kmeans', 'kmeans_regimes'); print(f'k terbaik = {KBEST}'); print(RG.to_string(index=False))
np.savez_compressed(os.path.join(DIR_DATA, 'P07_peta.npz'), X0=X0, Y0=Y0, RES=RES, crs=CRS, sen_FNU_per_thn=SENMAP, tren_signifikan=SIGMAP, rezim=REG,
                    median_hujan=np.nanmedian(TL[[k for k, p in enumerate(PL) if p.month in WET]], axis=0), median_kemarau=np.nanmedian(TL[[k for k, p in enumerate(PL) if p.month in DRY]], axis=0))
DF.drop(columns='p').round(4).to_csv(os.path.join(DIR_DATA, 'P07_bulanan_hujan_plume.csv'), index=False)

## Tahap 9 — Gambar dua bahasa dan unduhan

In [ ]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, ListedColormap, TwoSlopeNorm
plt.rcParams.update({'font.size': 9}); ext = [0, W * RES / 1000, 0, H * RES / 1000]; mxk, myk = (cxm + 0.5) * RES / 1000, (H - cym - 0.5) * RES / 1000
ZC = ['#d7301f', '#fc8d59', '#4575b4']
T_ = {'ID': dict(wet='Musim hujan (Des–Apr)', dry='Musim kemarau (Jun–Okt)', ratio='Rasio hujan / kemarau', fnu='Kekeruhan (FNU)', km='km', mouth='muara Maro', rain='Hujan DAS (mm/bulan)',
                 yr='Tahun', mon='Bulan', sen='Kemiringan Sen (FNU/tahun)', sig='titik = signifikan (p < 0,05)', reg='Rezim', ls='Landsat 8/9 (FNU)', s2='Sentinel-2 (FNU)', med='Median tahunan (FNU)'),
      'EN': dict(wet='Wet season (Dec–Apr)', dry='Dry season (Jun–Oct)', ratio='Wet / dry ratio', fnu='Turbidity (FNU)', km='km', mouth='Maro mouth', rain='Catchment rain (mm/month)',
                 yr='Year', mon='Month', sen="Sen's slope (FNU/year)", sig='dots = significant (p < 0.05)', reg='Regime', ls='Landsat 8/9 (FNU)', s2='Sentinel-2 (FNU)', med='Annual median (FNU)')}
MW, MD = np.load(os.path.join(DIR_DATA, 'P07_peta.npz'))['median_hujan'], np.load(os.path.join(DIR_DATA, 'P07_peta.npz'))['median_kemarau']
def rings(ax):
    for z in ZONES: ax.add_patch(plt.Circle((mxk, myk), z[1], fill=False, ls='--', lw=0.6, color='k'))
    ax.plot(mxk, myk, 'k*', ms=8); ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3])
for lang, t in T_.items():
    fig, axs = plt.subplots(1, 3, figsize=(16, 4.6), gridspec_kw={'wspace': 0.35})
    for ax, A_, ttl in ((axs[0], MW, t['wet']), (axs[1], MD, t['dry'])):
        im = ax.imshow(np.where(WATER, A_, np.nan), norm=LogNorm(1, 1000), cmap='YlOrBr', extent=ext); rings(ax); ax.set_title(ttl); ax.set_xlabel(t['km'])
    fig.colorbar(im, ax=axs[:2], fraction=0.025, label=t['fnu'])
    im2 = axs[2].imshow(np.where(WATER, MW / MD, np.nan), norm=LogNorm(0.25, 4), cmap='RdBu_r', extent=ext); rings(axs[2]); axs[2].set_title(t['ratio']); fig.colorbar(im2, ax=axs[2], fraction=0.04)
    fig.savefig(os.path.join(DIR_FIG, f'P07_gambar_peta_musim_{lang}.png'), dpi=250, bbox_inches='tight'); plt.close(fig)
    fig, ax = plt.subplots(figsize=(12, 4)); ax2 = ax.twinx(); ax2.bar(RAIN.p.dt.to_timestamp(), RAIN.hujan_mm, width=25, color='#9ecae1', alpha=0.6); ax2.set_ylabel(t['rain'])
    for z, c in zip(ZONES, ZC):
        ax.plot(ZL.p.dt.to_timestamp(), ZL[ZNAME[z]], '-', color=c, lw=1, label=f'{ZNAME[z]} (Landsat)')
        ax.plot(ZS.p.dt.to_timestamp(), ZS[ZNAME[z]], '.', color=c, ms=3, label=f'{ZNAME[z]} (S2)')
    ax.set_yscale('log'); ax.set_ylabel(t['fnu']); ax.set_zorder(ax2.get_zorder() + 1); ax.patch.set_visible(False); ax.legend(fontsize=7, ncol=3); ax.set_xlabel(t['yr'])
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'P07_gambar_deret_waktu_{lang}.png'), dpi=300); plt.close(fig)
    fig, ax = plt.subplots(figsize=(7, 4)); ax2 = ax.twinx(); rm = RAIN.groupby('bulan').hujan_mm.mean(); ax2.bar(rm.index, rm.values, color='#9ecae1', alpha=0.6); ax2.set_ylabel(t['rain'])
    for z, c in zip(ZONES, ZC):
        g = ZL.groupby('bulan')[ZNAME[z]]; ax.plot(g.median().index, g.median().values, 'o-', color=c, label=ZNAME[z]); ax.fill_between(g.median().index, g.quantile(0.25), g.quantile(0.75), color=c, alpha=0.15)
    ax.set_yscale('log'); ax.set_xticks(range(1, 13)); ax.set_xlabel(t['mon']); ax.set_ylabel(t['fnu']); ax.set_zorder(ax2.get_zorder() + 1); ax.patch.set_visible(False); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'P07_gambar_siklus_musim_{lang}.png'), dpi=300); plt.close(fig)
    fig, axs = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={'width_ratios': [1.2, 1]})
    vm = np.nanpercentile(np.abs(SENMAP), 98) if np.isfinite(SENMAP).any() else 1
    im = axs[0].imshow(SENMAP, cmap='RdBu_r', norm=TwoSlopeNorm(0, -vm, vm), extent=ext); yy_, xx_ = np.where(SIGMAP[::6, ::6]); rings(axs[0])
    axs[0].plot((xx_ * 6 + 0.5) * RES / 1000, (H - yy_ * 6 - 0.5) * RES / 1000, 'k.', ms=1); fig.colorbar(im, ax=axs[0], fraction=0.04, label=t['sen']); axs[0].set_title(t['sig']); axs[0].set_xlabel(t['km'])
    for z, c in zip(ZONES, ZC): axs[1].plot(AN.index, AN[ZNAME[z]], 'o-', color=c, label=ZNAME[z])
    axs[1].set_yscale('log'); axs[1].set_xlabel(t['yr']); axs[1].set_ylabel(t['med']); axs[1].legend(fontsize=8); axs[1].grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'P07_gambar_tren_{lang}.png'), dpi=250); plt.close(fig)
    fig, axs = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={'width_ratios': [1.2, 1]}); cols = plt.cm.viridis(np.linspace(0, 1, KBEST))
    axs[0].imshow(np.where(REG > 0, REG, np.nan), cmap=ListedColormap(cols), vmin=0.5, vmax=KBEST + 0.5, extent=ext, interpolation='nearest'); rings(axs[0]); axs[0].set_xlabel(t['km'])
    for r in range(1, KBEST + 1): axs[1].plot(range(1, 13), 10 ** km.cluster_centers_[order[r - 1]], 'o-', color=cols[r - 1], label=f'{t["reg"]} {r}')
    axs[1].set_yscale('log'); axs[1].set_xticks(range(1, 13)); axs[1].set_xlabel(t['mon']); axs[1].set_ylabel(t['fnu']); axs[1].legend(fontsize=8)
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'P07_gambar_rezim_kmeans_{lang}.png'), dpi=250); plt.close(fig)
    fig, ax = plt.subplots(figsize=(5, 5)); sel = np.random.default_rng(1).choice(len(xs_), size=min(20000, len(xs_)), replace=False) if len(xs_) else []
    if len(xs_): ax.hexbin(xs_[sel], ys_[sel], gridsize=60, xscale='log', yscale='log', bins='log', cmap='Greys', mincnt=1); ax.plot([0.5, 3000], [0.5, 3000], 'r--', lw=1)
    ax.set_xlabel(t['ls']); ax.set_ylabel(t['s2']); ax.set_title(f'ρ = {rho:.2f}'); fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'P07_gambar_landsat_vs_s2_{lang}.png'), dpi=300); plt.close(fig)
print(sorted(os.listdir(DIR_FIG)))

In [ ]:
summary = {'muara': META['mouth'], 'DAS_km2': META['catch_km2'], 'ambang_plume_FNU': T_PLUME, 'piksel_air_zona': {ZNAME[z]: int(m.sum()) for z, m in ZMASK.items()},
           'data_per_tahun': DY.to_dict('records'), 'kesesuaian_sensor': CMP.to_dict('records'), 'musim': SEA.to_dict('records'), 'tren': TREND.to_dict('records'),
           'tren_piksel': PIXT.to_dict('records'), 'korelasi_hujan': COR.to_dict('records'), 'RF': RFT.to_dict('records'), 'kepentingan_RF': IMPT.to_dict('records'),
           'silhouette': SIL.to_dict('records'), 'rezim': RG.to_dict('records')}
json.dump(summary, open(os.path.join(DIR_TAB, 'P07_hasil_ringkas.json'), 'w'), indent=2, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
with pd.ExcelWriter(os.path.join(DIR_TAB, 'P07_semua_tabel.xlsx')) as xw:
    for f in sorted(glob.glob(os.path.join(DIR_TAB, '*.csv'))): pd.read_csv(f).to_excel(xw, sheet_name=os.path.basename(f)[4:-4][:31], index=False)
zpath = '/content/P07_hasil.zip' if os.path.exists('/content') else 'P07_hasil.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fs_ in os.walk(OUT):
        for fn in fs_: z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), os.path.dirname(OUT)))
import shutil; shutil.copy(zpath, os.path.join(BASE, 'P07_hasil.zip'))
print(SEA.to_string(index=False)); print(TREND.to_string(index=False))
try:
    from google.colab import files; files.download(zpath)
except Exception:
    print('Arsip:', zpath)
print('Salinan juga tersimpan di Drive:', os.path.join(BASE, 'P07_hasil.zip'))

Selesai. Upload `P07_hasil.zip` ke Claude. Salinannya juga ada di Google Drive `P07_kekeruhan/P07_hasil.zip`.